# Sentiment-Analyse der Kommentare mit RoBERTa

Dieses Notebook analysiert die TikTok-Kommentare mit einem RoBERTa-basierten Sentiment-Modell.

Ziele:
- Sentiment-Labels und Wahrscheinlichkeiten pro Kommentar bestimmen
- KI- und Real-Videos auf Kommentar-Ebene und auf Video-Ebene vergleichen
- den Zusammenhang zwischen Kommentar-Sentiment und Engagement-Rate prüfen
- auswertbare CSV-Dateien für Ergebnisdarstellung und Anhang exportieren


In [1]:
from pathlib import Path
from math import sqrt

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, pipeline

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

COLOR_AI = "#4C78A8"
COLOR_REAL = "#F5855B"
PALETTE = {"KI": COLOR_AI, "Real": COLOR_REAL}


In [ ]:
BASE_DIR = Path.cwd().resolve().parents[1]
DATA_PATH = BASE_DIR / "data" / "01_raw" / "comments_metadata" / "02_AI_AND_REAL_TIKTOK_COMMENTS_DATASET_2025.csv"
VIDEO_DATA_PATH = BASE_DIR / "data" / "01_raw" / "videos_metadata" / "01_AI_AND_REAL_TIKTOK_VIDEOS_DATASET_2025.csv"
AI_VIDEOS_RAW_PATH = BASE_DIR / "data" / "01_raw" / "videos_metadata" / "01_AI_TIKTOK_VIDEOS_DATASET_2025.csv"
REAL_VIDEOS_RAW_PATH = BASE_DIR / "data" / "01_raw" / "videos_metadata" / "01_REAL_TIKTOK_VIDEOS_DATASET_2025.csv"
OUTPUT_DIR = BASE_DIR / "comments" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
BATCH_SIZE = 64
TEXT_COLUMN = "comment_text"

FILTER_ENGLISH_ONLY = True
KEEP_REPLIES = True

COMMENT_RESULTS_CSV = OUTPUT_DIR / "02_comment_sentiment_results.csv"
VIDEO_LEVEL_CSV = OUTPUT_DIR / "02_comment_sentiment_video_level.csv"
MISSING_ENGAGEMENT_EXCLUDED_CSV = OUTPUT_DIR / "02_comment_sentiment_excluded_missing_engagement.csv"
MISSING_VIDEO_METADATA_CSV = OUTPUT_DIR / "02_comment_sentiment_missing_video_metadata.csv"
ENGAGEMENT_SOURCE_CSV = OUTPUT_DIR / "02_comment_sentiment_engagement_sources.csv"

print(f"Kommentar-Datensatz: {DATA_PATH}")
print(f"Video-Datensatz: {VIDEO_DATA_PATH}")
print(f"Ausgabeordner: {OUTPUT_DIR}")
print(f"Modell: {MODEL_NAME}")


Kommentar-Datensatz: /Users/hannahernstberger/Documents/Master_/data/01_raw/comments_metadata/02_AI_AND_REAL_TIKTOK_COMMENTS_DATASET_2025.csv
Video-Datensatz: /Users/hannahernstberger/Documents/Master_/data/01_raw/videos_metadata/01_AI_AND_REAL_TIKTOK_VIDEOS_DATASET_2025.csv
Ausgabeordner: /Users/hannahernstberger/Documents/Master_/comments/results
Modell: cardiffnlp/twitter-roberta-base-sentiment-latest


In [ ]:
keep_cols = [
    "comment_id",
    "video_id",
    "influencer_type",
    "comment_language",
    "is_reply_to_comment",
    "comment_like_count",
    "comment_reply_count",
    "video_caption",
    TEXT_COLUMN,
]

comments_df = pd.read_csv(DATA_PATH, usecols=keep_cols).copy()
initial_n = len(comments_df)

# Engagement wird zuerst aus dem kombinierten Video-Datensatz gelesen.
# Fehlende Video-IDs werden danach aus den Roh-Video-Dateien rekonstruiert.
comments_df["video_id"] = comments_df["video_id"].astype("string").str.strip()
video_meta = pd.read_csv(
    VIDEO_DATA_PATH,
    usecols=["video_id", "video_engagement_rate"],
).drop_duplicates(subset="video_id")
video_meta["video_id"] = video_meta["video_id"].astype("string").str.strip()
video_meta["engagement_source"] = "combined_video_dataset"

raw_video_meta_frames = []
for raw_path, raw_type in [(AI_VIDEOS_RAW_PATH, "ai_raw_video_dataset"), (REAL_VIDEOS_RAW_PATH, "real_raw_video_dataset")]:
    raw_meta = pd.read_csv(raw_path, usecols=["id", "likes", "comments", "shares", "plays"])
    raw_meta = raw_meta.rename(columns={"id": "video_id"})
    raw_meta["video_id"] = raw_meta["video_id"].astype("string").str.strip()
    for metric_col in ["likes", "comments", "shares", "plays"]:
        raw_meta[metric_col] = pd.to_numeric(raw_meta[metric_col], errors="coerce")
    raw_meta["video_engagement_rate"] = (
        raw_meta["likes"].fillna(0) + raw_meta["comments"].fillna(0) + raw_meta["shares"].fillna(0)
    ) / raw_meta["plays"].replace(0, np.nan)
    raw_meta["engagement_source"] = raw_type
    raw_video_meta_frames.append(raw_meta[["video_id", "video_engagement_rate", "engagement_source"]])
raw_video_meta = pd.concat(raw_video_meta_frames, ignore_index=True).drop_duplicates(subset="video_id")

video_meta_complete = pd.concat([video_meta, raw_video_meta], ignore_index=True)
video_meta_complete = video_meta_complete.dropna(subset=["video_engagement_rate"])
video_meta_complete = video_meta_complete.drop_duplicates(subset="video_id", keep="first")
comments_df = comments_df.merge(video_meta_complete, on="video_id", how="left")
merged_n = len(comments_df)

# Doppelte Kommentar-IDs würden dieselbe Beobachtung mehrfach zählen und werden deshalb entfernt.
comments_df = comments_df.drop_duplicates(subset="comment_id").copy()
dedup_n = len(comments_df)

# Kommentartexte werden in ein einheitliches Format gebracht, damit das Modell keine Leerstring-Artefakte bekommt.
comments_df[TEXT_COLUMN] = (
    comments_df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
comments_df = comments_df[comments_df[TEXT_COLUMN] != ""].copy()
after_text_clean_n = len(comments_df)

if FILTER_ENGLISH_ONLY and "comment_language" in comments_df.columns:
    comments_df = comments_df[comments_df["comment_language"].fillna("").str.lower().eq("en")].copy()
after_language_filter_n = len(comments_df)

if not KEEP_REPLIES and "is_reply_to_comment" in comments_df.columns:
    comments_df = comments_df[comments_df["is_reply_to_comment"].fillna("no").str.lower().ne("yes")].copy()
after_reply_filter_n = len(comments_df)

missing_video_metadata_df = (
    comments_df[comments_df["video_engagement_rate"].isna()]
    .groupby(["video_id", "influencer_type"], dropna=False)
    .agg(
        comment_count=("comment_id", "count"),
        example_comment=(TEXT_COLUMN, "first"),
    )
    .reset_index()
    .sort_values("comment_count", ascending=False)
)
excluded_missing_engagement_df = comments_df[comments_df["video_engagement_rate"].isna()].copy()
comments_df = comments_df[comments_df["video_engagement_rate"].notna()].copy()
after_engagement_filter_n = len(comments_df)
missing_engagement_before_filter = len(excluded_missing_engagement_df)
missing_engagement_video_n = excluded_missing_engagement_df["video_id"].nunique()
missing_engagement = comments_df["video_engagement_rate"].isna().sum()
engagement_source_video_df = (
    comments_df.groupby(["video_id", "influencer_type", "engagement_source", "video_engagement_rate"], dropna=False)
    .size()
    .rename("comment_count")
    .reset_index()
    .sort_values(["engagement_source", "comment_count"], ascending=[True, False])
)

comments_df["Typ"] = comments_df["influencer_type"].map({"ai": "KI", "real": "Real"}).fillna(comments_df["influencer_type"])
comments_df["comment_length_chars"] = comments_df[TEXT_COLUMN].str.len()
comments_df["comment_length_words"] = comments_df[TEXT_COLUMN].str.split().str.len()

print(f"Kommentare vor Bereinigung: {initial_n:,}")
print(f"Nach Engagement-Merge: {merged_n:,}")
print(f"Nach Entfernen doppelter comment_id: {dedup_n:,}")
print(f"Nach Textbereinigung: {after_text_clean_n:,}")
print(f"Nach Sprachfilter: {after_language_filter_n:,}")
print(f"Nach Reply-Filter: {after_reply_filter_n:,}")
print(f"Ausgeschlossen wegen fehlender Engagement-Rate: {missing_engagement_before_filter:,} Kommentare aus {missing_engagement_video_n:,} Videos")
print(f"Kommentare in der Analyse: {after_engagement_filter_n:,}")
print(f"Kommentare ohne gematchte Engagement-Rate nach Filter: {missing_engagement:,}")
display(missing_video_metadata_df.head(20))
display(comments_df.head(3))


Kommentare vor Bereinigung: 80,817
Nach Engagement-Merge: 80,817
Nach Entfernen doppelter comment_id: 80,817
Nach Textbereinigung: 80,497
Nach Sprachfilter: 39,615
Nach Reply-Filter: 39,615
Ausgeschlossen wegen fehlender Engagement-Rate: 6,944 Kommentare aus 83 Videos
Kommentare in der Analyse: 32,671
Kommentare ohne gematchte Engagement-Rate nach Filter: 0


,video_id,influencer_type,comment_count,example_comment
3,7208351443193974062,real,1787,If Niall Horan looked at me like that I’d forg...
6,7371613331704139040,real,1412,Julia Zelg for me. I couldn’t recognize her wh...
2,7040984071631228206,real,853,So y’all rich rich. Huh. Are you looking to ad...
1,7014140703680826629,real,701,I don’t have a nice side profile like u girl :/
15,7538581166312426774,real,374,"go to David's salon, your $3 would be so much ..."
4,7320000759762881824,real,357,where is your coat from
25,7578555804157103391,real,179,Why are people bullying a child who is just ha...
7,7383032827526450478,real,170,books and pilates are real hobbies don’t let m...
5,7364519121855319339,real,62,this would heal me
0,6869412214848163077,real,61,the fact that this has 6.7M likes


,comment_id,comment_text,comment_like_count,comment_reply_count,video_id,video_caption,is_reply_to_comment,comment_language,influencer_type,video_engagement_rate,engagement_source,Typ,comment_length_chars,comment_length_words
1178,7541804977875550983,Wow I liked 💗,1,0,7541766006897773838,Every corner of the Rock Bay Villas feels like...,no,en,real,0.020297,combined_video_dataset,Real,13,4
1192,7543754834214945557,Need this therapy too✨,0,0,7541766006897773838,Every corner of the Rock Bay Villas feels like...,no,en,real,0.020297,combined_video_dataset,Real,22,4
1197,7572978399367676686,and the little sister too,1,0,7571506841398562078,"#ad #Ad Tonight’s agenda: snacks, holiday movi...",yes,en,real,0.046320,combined_video_dataset,Real,25,5


In [ ]:
prep_steps = [
    ("Ausgangsdatensatz", initial_n),
    ("Nach Engagement-Merge", merged_n),
    ("Nach Entfernen doppelter comment_id", dedup_n),
    ("Nach Entfernen leerer Kommentare", after_text_clean_n),
]
if FILTER_ENGLISH_ONLY:
    prep_steps.append(("Nach Sprachfilter (nur Englisch)", after_language_filter_n))
else:
    prep_steps.append(("Ohne Sprachfilter", after_language_filter_n))
if KEEP_REPLIES:
    prep_steps.append(("Replies beibehalten", after_reply_filter_n))
else:
    prep_steps.append(("Nach Reply-Filter", after_reply_filter_n))
prep_steps.append(("Ausgeschlossen: keine Engagement-Rate", missing_engagement_before_filter))
prep_steps.append(("Videos ohne gematchte Engagement-Rate", missing_engagement_video_n))
prep_steps.append(("Finale Analyse mit Engagement-Rate", after_engagement_filter_n))

prep_overview = pd.DataFrame(prep_steps, columns=["Schritt", "Kommentare"])
display(prep_overview)
display(comments_df["engagement_source"].value_counts().rename_axis("engagement_source").reset_index(name="comments"))

structure_df = (
    comments_df.groupby("Typ")
    .agg(
        Kommentare=("comment_id", "count"),
        Videos=("video_id", "nunique"),
        Kommentarzeichen_M=("comment_length_chars", "mean"),
        Kommentarwoerter_M=("comment_length_words", "mean"),
        Kommentare_ohne_Engagement=("video_engagement_rate", lambda s: s.isna().sum()),
    )
    .round(2)
    .reset_index()
)
display(structure_df)


,Schritt,Kommentare
0,Ausgangsdatensatz,80817
1,Nach Engagement-Merge,80817
2,Nach Entfernen doppelter comment_id,80817
3,Nach Entfernen leerer Kommentare,80497
4,Nach Sprachfilter (nur Englisch),39615
5,Replies beibehalten,39615
6,Ausgeschlossen: keine Engagement-Rate,6944
7,Videos ohne gematchte Engagement-Rate,83
8,Finale Analyse mit Engagement-Rate,32671


,engagement_source,comments
0,combined_video_dataset,30967
1,ai_raw_video_dataset,1466
2,real_raw_video_dataset,238


,Typ,Kommentare,Videos,Kommentarzeichen_M,Kommentarwoerter_M,Kommentare_ohne_Engagement
0,KI,14576,238,32.13,6.45,0
1,Real,18095,640,72.12,13.89,0


In [ ]:
if torch.cuda.is_available():
    pipeline_device = 0
    device_label = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    pipeline_device = "mps"
    device_label = "mps"
else:
    pipeline_device = -1
    device_label = "cpu"

print(f"Verwendetes Device für Kommentar-Sentiment: {device_label}")

max_length = 512
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, model_max_length=max_length)
sentiment_pipe = pipeline(
    task="sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=tokenizer,
    device=pipeline_device,
    truncation=True,
    max_length=max_length,
    top_k=None,
)

texts = comments_df[TEXT_COLUMN].tolist()
all_results = []
for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="RoBERTa-Sentiment Kommentare"):
    batch = texts[start:start + BATCH_SIZE]
    batch_results = sentiment_pipe(batch, truncation=True, padding=True, max_length=max_length)
    all_results.extend(batch_results)

rows = []
for text, result in zip(texts, all_results):
    scores = {entry["label"].lower(): entry["score"] for entry in result}
    dominant = max(scores, key=scores.get)
    sentiment_index = scores.get("positive", 0.0) - scores.get("negative", 0.0)
    rows.append(
        {
            TEXT_COLUMN: text,
            "negative": scores.get("negative", 0.0),
            "neutral": scores.get("neutral", 0.0),
            "positive": scores.get("positive", 0.0),
            "sentiment_label": dominant,
            "sentiment_score_max": scores.get(dominant, 0.0),
            "sentiment_index": sentiment_index,
            "sentiment_label_de": {"negative": "negativ", "neutral": "neutral", "positive": "positiv"}.get(dominant, dominant),
        }
    )

sentiment_df = pd.DataFrame(rows)
comments_df = pd.concat([comments_df.reset_index(drop=True), sentiment_df], axis=1)
comments_df.head()




def dominant_value(series):
    mode = series.mode(dropna=True)
    return mode.iloc[0] if not mode.empty else np.nan

video_level_df = (
    comments_df.groupby(["video_id", "Typ", "video_engagement_rate"], dropna=False)
    .agg(
        comment_count=("comment_id", "count"),
        sentiment_index_mean=("sentiment_index", "mean"),
        sentiment_index_median=("sentiment_index", "median"),
        dominant_comment_sentiment=("sentiment_label_de", dominant_value),
    )
    .reset_index()
)



Verwendetes Device für Kommentar-Sentiment: mps


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps


RoBERTa-Sentiment Kommentare:   0%|          | 0/511 [00:00<?, ?it/s]

In [ ]:
comments_df.to_csv(COMMENT_RESULTS_CSV, index=False)
video_level_df.to_csv(VIDEO_LEVEL_CSV, index=False)
excluded_missing_engagement_df.to_csv(MISSING_ENGAGEMENT_EXCLUDED_CSV, index=False)
missing_video_metadata_df.to_csv(MISSING_VIDEO_METADATA_CSV, index=False)
engagement_source_video_df.to_csv(ENGAGEMENT_SOURCE_CSV, index=False)
print(f"Gespeichert: {COMMENT_RESULTS_CSV}")
print(f"Gespeichert: {VIDEO_LEVEL_CSV}")
